In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# 设置绘图风格
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

# 1. 加载数据
print("正在加载数据...")
df = pd.read_csv('../data/kg.csv')

# PrimeKG 的典型格式通常是: head_type::head_id, relation_type::relation_id, tail_type::tail_id
# 如果列名不是默认的 0, 1, 2，请根据实际情况修改
df.columns = ['head', 'relation', 'tail']

print(f"数据加载完成。总三元组数量: {len(df)}")

# 2. 解析节点和关系类型
# 假设格式为 "type::id"，我们需要提取冒号前的部分
def get_type(x):
    return x.split('::')[0]

df['head_type'] = df['head'].apply(get_type)
df['tail_type'] = df['tail'].apply(get_type)
# 关系通常不需要分类型，或者其类型就包含在关系名中，这里我们直接看关系名

print("\n--- 1. 节点类型分布 ---")
# 统计所有出现的节点类型
head_types = df['head_type'].value_counts()
tail_types = df['tail_type'].value_counts()
all_types = pd.concat([head_types, tail_types]).sort_values(ascending=False)
print(all_types)

print("\n--- 2. 关系(边)类型分布 ---")
# 统计关系类型的数量
relation_counts = df['relation'].value_counts()
print(f"共有 {len(relation_counts)} 种不同的关系类型。")
print("前 10 种最频繁的关系:")
print(relation_counts.head(10))

# 3. 构建异构图连接矩阵 (用于分析元路径可行性)
print("\n--- 3. 异构图连接模式 (Head -> Relation -> Tail) ---")
# 我们看看哪些类型之间是有连边的
connection_patterns = df.groupby(['head_type', 'relation', 'tail_type']).size().reset_index(name='count')
# 按连接数量排序
connection_patterns = connection_patterns.sort_values('count', ascending=False)
print("前 20 种连接模式:")
print(connection_patterns.head(20))

# 4. 可视化：关系类型频率 (Top 20)
plt.figure(figsize=(15, 8))
top_relations = relation_counts.head(20)
sns.barplot(x=top_relations.index, y=top_relations.values, palette="viridis")
plt.title('Top 20 Relation Types in PrimeKG', fontsize=16)
plt.xlabel('Relation Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 5. 可视化：节点类型数量分布
plt.figure(figsize=(15, 8))
sns.barplot(x=all_types.index, y=all_types.values, palette="magma")
plt.title('Node Type Distribution', fontsize=16)
plt.xlabel('Node Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 6. 简单的网络概览 (只画类型级别的图)
# 创建一个简单的图，节点是"类型"，边表示这两个类型之间有连接
G = nx.DiGraph()
for _, row in connection_patterns.iterrows():
    G.add_edge(row['head_type'], row['tail_type'], weight=row['count'])

plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.5, iterations=50)
# 根据边的权重画线宽
edges = G.edges()
weights = [G[u][v]['weight'] / 1000 for u, v in edges] # 缩放线宽以便观察

nx.draw_networkx_nodes(G, pos, node_size=2000, node_color='skyblue', alpha=0.8)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
nx.draw_networkx_edges(G, pos, width=weights, edge_color='gray', arrows=True, arrowsize=20, alpha=0.6)

plt.title('PrimeKG Schema: Connections between Node Types', fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()